## Neighbourhood simulation
Where does a sound-driven signal actually go in the real wiring, when the path is not picked in advance?

---

### Init

In [ ]:
import pandas as pd
from pathlib import Path
import pyarrow.dataset as ds
import pyarrow.compute as pc
import duckdb
import networkx as nx

root = Path.cwd().parent
# root = Path(__file__).resolve().parent.parent

neurons_file = root / 'data' / 'body-annotations-male-cns-v1.0-minconf-0.5.feather'
connections_file = root / 'data' / 'connectome-weights-male-cns-v1.0-minconf-0.5.feather'

df = pd.read_feather(neurons_file) # directly to pandas dataframe
traced = df[df['status'] == 'Traced']
id_to_superclass = traced.set_index('bodyId')['superclass'].to_dict()

# lazy handle on the big file, nothing is loaded until a filtered read asks for it
dataset = ds.dataset(connections_file, format="feather")


# common functions and lookups copied from the earlier sections of 05

# get outgoing connections of specified neurons
def query_edges(neuron_ids, min_weight=3):
    if not isinstance(neuron_ids, (list, set, tuple)):
        neuron_ids = [neuron_ids]
    expr = pc.field("body_pre").isin(neuron_ids) & (pc.field("weight") >= min_weight)
    return dataset.to_table(filter=expr).to_pandas()

def grow_neighbourhood(start_ids, n_hops, min_weight=3):
    if not isinstance(start_ids, (list, set, tuple)):
        start_ids = [start_ids]

    seen = set(start_ids)       # every neuron reached so far, starting ones included
    to_check = set(start_ids)   # neurons whose outgoing connections we look up this hop
    all_edges = []

    for hop in range(1, n_hops + 1):
        hop_edges = query_edges(to_check, min_weight=min_weight)
        all_edges.append(hop_edges)

        to_check = set(hop_edges['body_post']) - seen   # targets not seen before
        # seen |= to_check
        seen.update(to_check)

        # prints each hop statistics
        print(f"hop {hop}: {len(hop_edges)} edges, {len(to_check)} new neurons, {len(seen)} total")
        if not to_check:
            break

    return pd.concat(all_edges, ignore_index=True), seen

def run_lif(edges, external_input, pulse_ticks, threshold, decay_factor, time_steps, verbose=True):
    # who each neuron sends charge to: {neuron: [(target, weight), ...]}
    outgoing = {}
    for pre, post, weight in edges[['body_pre', 'body_post', 'weight']].itertuples(index=False):
        outgoing.setdefault(pre, []).append((post, weight))

    charge = {}   # only neurons currently holding charge; a missing neuron means 0
    events = []   # (tick, neuron) for every firing

    for t in range(time_steps):
        if t in pulse_ticks:
            if verbose:
                print(f"tick {t}: external input added to {len(external_input)} neuron(s)")
            for nid, amount in external_input.items():
                charge[nid] = charge.get(nid, 0) + amount

        # who fires this tick, judged on charge before anything resets
        fired = [nid for nid, c in charge.items() if c >= threshold]

        # leak for everyone
        for nid in charge:
            charge[nid] *= decay_factor

        # firing neurons pass their weights on, then reset
        for nid in fired:
            events.append((t, nid))
            for post, weight in outgoing.get(nid, []):
                charge[post] = charge.get(post, 0) + weight
        for nid in fired:
            charge.pop(nid, None)
            if verbose and nid in motor_info:
                info = motor_info[nid]
                print(f"tick {t}: vnc_motor {nid} fired ({info['somaNeuromere']}, {info['instance']})")

    return events # returns a set of tick, id

motor_info = (traced[traced['superclass'] == 'vnc_motor']
              .set_index('bodyId')[['somaNeuromere', 'instance']]
              .fillna({'somaNeuromere': 'unknown'}) # added missing values
              .to_dict('index'))

from collections import Counter

totals = Counter(info['somaNeuromere'] for info in motor_info.values())

# regrow neighbourhood again
jo_neurons = traced[traced['type'].str.contains('JO', na=False)]

jo_auditory = jo_neurons[jo_neurons['type'].str.contains('JO-A|JO-B', na=False, regex=True)]

jo_ids = set(jo_auditory['bodyId'])

edges_jo, seen_jo = grow_neighbourhood(jo_ids, n_hops=4, min_weight=10)

leg_segments = ['T1', 'T2', 'T3']

abdomen_segments = [f'A{i}' for i in range(1, 11)]

def group_fraction(segments, group):
    return sum(segments[s] for s in group) / sum(totals[s] for s in group)

sources = {nid: 10_000 for nid in jo_ids}

##### Modifying the Tick Amount

**goal:** check that the result is real and not just a side effect of how long the simulation ran, then explain it.
the earlier 20-tick runs may have stopped the signal partway, so this reruns at longer lengths. it then looks for the sharp change between a threshold where the signal floods the network and one where it dies out, finds the neuron that decides it, and compares how much incoming wiring leg and abdomen motor neurons have, to see why one group fires more than the other.

in the previous output, the first motor tick at 360 and 361 is 8 and 9, and each run lasts 20 ticks. the legs and abdomen may be reached at different speeds, so 61% against 28% could partly reflect which group the clock allowed

rerun with longer runs and see whether the fractions move:

In [25]:
def report(events, label):
    fired_ids = {nid for t, nid in events}
    motor_ids = {nid for nid in fired_ids if nid in motor_info}
    first_motor = min((t for t, nid in events if nid in motor_info), default=None)
    segments = Counter(motor_info[nid]['somaNeuromere'] for nid in motor_ids)
    print(f"{label}: {len(fired_ids)} fired, "
          f"legs {group_fraction(segments, leg_segments):.0%}, "
          f"abdomen {group_fraction(segments, abdomen_segments):.0%}, "
          f"first motor tick {first_motor}")

for steps in [20, 40, 80]:
    events = run_lif(edges_jo, sources, {0}, 360, 0.8, steps, verbose=False)
    report(events, f"threshold 360, {steps} ticks")

threshold 360, 20 ticks: 7914 fired, legs 61%, abdomen 29%, first motor tick 9
threshold 360, 40 ticks: 33987 fired, legs 75%, abdomen 89%, first motor tick 9
threshold 360, 80 ticks: 34979 fired, legs 75%, abdomen 91%, first motor tick 9


> back to square one, the "leg-selective" conclusion at 360/361 wasn’t real. it only appeared because the LIF model stopped too early

next, try to re-scan the high thresholds at a higher tick amount:

In [26]:
thresholds = [310, 320, 330, 340, 350, 360, 361, 362, 363, 364, 365,
              370, 380, 390, 400, 600, 1000, 1500, 2500]

for th in thresholds:
    events = run_lif(edges_jo, sources, pulse_ticks={0}, threshold=th,
                     decay_factor=0.8, time_steps=80, verbose=False)
    report(events, f"threshold {th}, 80 ticks")

threshold 310, 80 ticks: 39892 fired, legs 77%, abdomen 93%, first motor tick 5
threshold 320, 80 ticks: 38776 fired, legs 77%, abdomen 93%, first motor tick 6
threshold 330, 80 ticks: 37778 fired, legs 76%, abdomen 93%, first motor tick 8
threshold 340, 80 ticks: 36833 fired, legs 76%, abdomen 93%, first motor tick 8
threshold 350, 80 ticks: 35974 fired, legs 76%, abdomen 92%, first motor tick 8
threshold 360, 80 ticks: 34979 fired, legs 75%, abdomen 91%, first motor tick 9
threshold 361, 80 ticks: 34819 fired, legs 75%, abdomen 91%, first motor tick 9
threshold 362, 80 ticks: 163 fired, legs 0%, abdomen 0%, first motor tick None
threshold 363, 80 ticks: 156 fired, legs 0%, abdomen 0%, first motor tick None
threshold 364, 80 ticks: 156 fired, legs 0%, abdomen 0%, first motor tick None
threshold 365, 80 ticks: 156 fired, legs 0%, abdomen 0%, first motor tick None
threshold 370, 80 ticks: 155 fired, legs 0%, abdomen 0%, first motor tick None
threshold 380, 80 ticks: 153 fired, legs 0%, 

> somewhere out there  theres a gate; so what happens is whatever's left in charge will just die out. i.e., above 362 the signal dies very early, which suggests a narrow gate near the start of the chain

the gate might not be a single neuron, so check the difference(s) in run 361 and 362:

In [27]:
common = dict(edges=edges_jo, external_input=sources, pulse_ticks={0},
              decay_factor=0.8, time_steps=80, verbose=False)

# run the simulation twice
events_361 = run_lif(threshold=361, **common)
events_362 = run_lif(threshold=362, **common)

 # keep neuron IDs from the 362 run as a set
fired_362 = {nid for t, nid in events_362}

# neurons that fired at 361 but never fired at all at 362
only_361 = [(t, nid) for t, nid in events_361 if nid not in fired_362] 

id_to_type = traced.set_index('bodyId')['type'].to_dict()
for t, nid in only_361[:15]:
    print(f"tick {t}: {nid}  {id_to_superclass.get(nid)}  {id_to_type.get(nid)}")

tick 5: 520628  cb_intrinsic  CB0591
tick 6: 10709  cb_intrinsic  WED193
tick 6: 12747  cb_intrinsic  CB0466
tick 6: 10141  descending_neuron  DNg15
tick 6: 12188  descending_neuron  DNg84
tick 6: 10038  descending_neuron  pIP1
tick 6: 12143  ascending_neuron  AN01A055
tick 6: 513114  ascending_neuron  AN01A055
tick 6: 10124  descending_neuron  DNg105
tick 7: 10263  cb_intrinsic  GNG300
tick 7: 529031  cb_intrinsic  GNG300
tick 7: 10313  descending_neuron  DNge031
tick 7: 523438  cb_intrinsic  SAD112_b
tick 7: 10218  descending_neuron  DNg35
tick 7: 11979  cb_motor  GNG651



> set subtraction works: at 362 the signal dies, so everything the 361 run reaches beyond that point is missing from fired_362.
>
> note: this was caught later on, but right now that code can't see a neuron that fires in both runs and fires an extra time in one.
>
> here tick is not hop count. CB0591 fires at tick 5, but it is not necessarily five synapses from the hearing neuron

but what is not proven yet is that CB0591 could be one of several gates that all sit just under 362, with CB0591 simply being the earliest

to find that out, list the incoming edges to 520628 from neurons that fire early and see if the ticks hover around 361-362:

In [28]:
gate = 520628

# 1) who feeds the gate
incoming = edges_jo[edges_jo['body_post'] == gate][['body_pre', 'weight']]
w_in = dict(zip(incoming['body_pre'], incoming['weight']))

first_fire = {}
for t, nid in events_361:
    if nid in w_in and nid not in first_fire:
        first_fire[nid] = t

for nid, w in sorted(w_in.items(), key=lambda kv: -kv[1]):
    print(f"{nid}  w={w}  {id_to_superclass.get(nid)}  {id_to_type.get(nid)}  first fired: {first_fire.get(nid)}")

# 2) replay the gate's charge
def gate_trace(events, label, last_tick=8, decay=0.8):
    print(label)
    charge = 0
    for t in range(last_tick + 1):
        print(f"  tick {t}: charge {charge:,.2f}")
        charge *= decay
        for t2, nid in events:
            if t2 == t and nid in w_in:
                charge += w_in[nid]

gate_trace(events_361, "threshold 361")
gate_trace(events_362, "threshold 362")

10731  w=199  cb_intrinsic  SAD107  first fired: 1
555500  w=168  descending_neuron  DNg29  first fired: 20
13883  w=133  ascending_neuron  AN17B005  first fired: 21
10739  w=117  descending_neuron  DNg24  first fired: 2
512690  w=90  cb_intrinsic  GNG516  first fired: 18
12747  w=83  cb_intrinsic  CB0466  first fired: 6
519538  w=82  descending_neuron  DNg24  first fired: 4
521298  w=82  descending_neuron  DNge130  first fired: 22
10925  w=54  cb_intrinsic  AVLP542  first fired: 18
13843  w=54  cb_intrinsic  GNG008  first fired: 20
10307  w=48  cb_intrinsic  AVLP083  first fired: 16
12950  w=48  cb_intrinsic  SAD057  first fired: None
10970  w=44  cb_intrinsic  GNG301  first fired: 22
10576  w=44  ascending_neuron  AN12B004  first fired: 22
522311  w=43  cb_intrinsic  GNG516  first fired: 18
10550  w=40  ascending_neuron  AN02A001  first fired: 17
18716  w=39  cb_intrinsic  CB3245  first fired: 23
18100  w=36  cb_intrinsic  CB3245  first fired: 22
34093  w=33  cb_intrinsic  AMMC018  f

> one limitation: this replay doesn't reset the charge when the gate fires, so the numbers after the firing tick are meaningless.

the gate wasn't a close call. at tick 5 its charge was 413 in the 361 run and 296 in the 362 run, so it does not hover around 361-362 as guessed above.

the two traces were identical up to tick 4 and then split. both had a charge of 267.84 at tick 4, and the charge at tick 5 is that value after the leak, plus whatever arrived at tick 4:
- 362 run: 267.84 * 0.8 = 214.27, plus an input of 82, which gives 296.27
- 361 run: 214.27 plus an input of 199, which gives 413.27

so the 361 run received 117 more at tick 4 (199 - 82).

neuron 10739 (DNg24) has weight 117, exactly the missing amount. it first fired at tick 2 in the list above, and the traces agree through tick 4, so the extra 117 at tick 4 would have to be a second firing.

but a 117 match is evidence rather than proof. the earlier subtraction compared neuron IDs only, so it could not see a neuron that fires in both runs but fires an extra time in one. comparing (tick, neuron) pairs can, so check directly:

In [29]:
set_361 = set(events_361)
set_362 = set(events_362)
diff = sorted(set_361 ^ set_362)   # pairs that appear in one run but not the other

for t, nid in diff[:10]:
    where = "361 only" if (t, nid) in set_361 else "362 only"
    print(f"tick {t}: {nid}  {where}  {id_to_superclass.get(nid)}  {id_to_type.get(nid)}")

tick 4: 10739  361 only  descending_neuron  DNg24
tick 5: 10128  361 only  cb_intrinsic  SAD103
tick 5: 11249  361 only  cb_intrinsic  WED190
tick 5: 520628  361 only  cb_intrinsic  CB0591
tick 6: 10038  361 only  descending_neuron  pIP1
tick 6: 10124  361 only  descending_neuron  DNg105
tick 6: 10141  361 only  descending_neuron  DNg15
tick 6: 10709  361 only  cb_intrinsic  WED193
tick 6: 10739  361 only  descending_neuron  DNg24
tick 6: 10944  361 only  cb_intrinsic  SAD098


> the first difference between the two runs is at tick 4, neuron 10739 (DNg24), which fires in the 361 run only. it also fired at tick 2 in both runs, so this is a second firing, as the 117 arithmetic predicted.
>
> everything after it follows from it: at tick 5 CB0591 (the earlier gate candidate), WED190 and SAD103 fire in the 361 run only, then at tick 6 the descending neurons pIP1, DNg105 and DNg15, and 10739 and SAD098 fire again.
>
> so CB0591 was not the gate, it was the first thing downstream of it. the real trigger is the second firing of 10739 at tick 4.

if 10739 fires at 361 and not at 362, its charge at tick 4 must be at least 361 and below 362. trace its charge to see:

In [30]:
def charge_trace(events, target, threshold, last_tick=6, decay=0.8):
    w_in = dict(zip(*[edges_jo.loc[edges_jo['body_post'] == target, c]
                      for c in ('body_pre', 'weight')]))
    charge = 0
    for t in range(last_tick + 1):
        fires = charge >= threshold
        print(f"  tick {t}: charge {charge:,.3f}" + ("   <- FIRES" if fires else ""))
        if fires:
            charge = 0          # run_lif drops the neuron's charge when it fires
        else:
            charge *= decay
            charge += sum(w_in[nid] for t2, nid in events if t2 == t and nid in w_in)

print("361 run"); charge_trace(events_361, 10739, 361)
print("362 run"); charge_trace(events_362, 10739, 362)

361 run
  tick 0: charge 0.000
  tick 1: charge 273.000
  tick 2: charge 545.400   <- FIRES
  tick 3: charge 0.000
  tick 4: charge 361.000   <- FIRES
  tick 5: charge 0.000
  tick 6: charge 361.000   <- FIRES
362 run
  tick 0: charge 0.000
  tick 1: charge 273.000
  tick 2: charge 545.400   <- FIRES
  tick 3: charge 0.000
  tick 4: charge 361.000
  tick 5: charge 288.800
  tick 6: charge 231.040


> neuron 10739 reaches a charge of exactly 361.000 at tick 4, in both runs. the traces are identical up to that point.
>
> - 361 run: 361 >= 361, so it fires again at tick 4 (and again at tick 6)
> - 362 run: 361 < 362, so it does not fire. its charge just leaks away (288.8, then 231.0)
>
> the charge is a whole number because the neuron resets to 0 when it fires at tick 2. nothing is left to leak by tick 3, so the 361 at tick 4 is only the sum of the weights arriving at tick 3.
>
> so the cliff sits where one neuron's summed input happens to equal 361. at 361 it fires a second time and keeps the downstream neurons going. at 362 that never starts, and the signal dies within about 25 neurons of the ear.

when modifying the threshold, there were two things to check:
- try to compare how much total incoming weight leg and abdominal motor neurons receive to see if they really are easier to excite
- extend the scan upward until the signal dies and see how big the gap is between legs and abdomen firing

previously the second was done, but now the first will be done because after experimenting with 80 ticks, a more accurate result can be observed:

In [31]:
# groups are done by legs (T1 to T3) and abdomen (A1 to A10)

incoming_total = edges_jo.groupby('body_post')['weight'].sum()

motor_in = pd.DataFrame(
    [(nid, info['somaNeuromere'], incoming_total.get(nid, 0)) for nid, info in motor_info.items()],
    columns=['bodyId', 'segment', 'incoming'])
motor_in['group'] = motor_in['segment'].map(
    lambda s: 'leg' if s in leg_segments else ('abdomen' if s in abdomen_segments else 'unknown'))

with pd.option_context('display.float_format', '{:,.1f}'.format):
    print(motor_in.groupby('group')['incoming'].describe(percentiles=[.25, .5, .75]))

# groups and counts, instead of back-calculating from a fraction
can_fire = motor_in.assign(can_fire=motor_in['incoming'] >= 361)
fire_summary = can_fire.groupby('group')['can_fire'].agg(['sum', 'count'])
fire_summary['fraction'] = fire_summary['sum'] / fire_summary['count']
print(fire_summary)

no_input = motor_in.assign(no_input=motor_in['incoming'] == 0)
no_input_summary = no_input.groupby('group')['no_input'].agg(['sum', 'count'])
no_input_summary['fraction'] = no_input_summary['sum'] / no_input_summary['count']
print(no_input_summary)

# the accumulation argument: a constant input each tick settles at 1 / (1 - decay) times its size
decay = 0.8
threshold = 361
print("settling multiplier:", 1 / (1 - decay))
print("per-tick input needed:", threshold * (1 - decay))

         count    mean     std   min     25%     50%     75%      max
group                                                                
abdomen  207.0 2,631.6 2,057.7   0.0 1,257.5 2,137.0 3,487.0  9,565.0
leg      500.0 2,875.5 3,553.5   0.0   163.5 1,235.5 4,812.8 16,748.0
unknown    1.0   719.0     NaN 719.0   719.0   719.0   719.0    719.0
         sum  count  fraction
group                        
abdomen  187    207  0.903382
leg      319    500  0.638000
unknown    1      1  1.000000
         sum  count  fraction
group                        
abdomen    4    207  0.019324
leg       32    500  0.064000
unknown    0      1  0.000000
settling multiplier: 5.000000000000001
per-tick input needed: 72.19999999999999


> total incoming weight per motor neuron (the most charge it can receive in one tick):
>
> | group | neurons | median | 25th percentile | max | no incoming edges | ceiling >= 361 |
> |---|---|---|---|---|---|---|
> | abdomen | 207 | 2,137 | 1,257 | 9,565 | 4 / 207 = 1.9% | 187 / 207 = 90.3% |
> | leg | 500 | 1,235 | 163 | 16,748 | 32 / 500 = 6.4% | 319 / 500 = 63.8% |
>
> the abdominal motor neurons almost all have large totals. the leg ones are very spread out: the median is lower, a quarter have a total under 164, and the mean (2,875) is pulled up by a few very high ones. this fits the abdomen being easier to excite in this model.
>
> but the ceiling is not an upper bound on firing. at threshold 361 the legs actually reached 75%, above their 63.8% ceiling (75 - 63.8 = 11.2 percentage points). the abdomen matched its ceiling almost exactly (91 - 90.3 = 0.7 points).
>
> why: charge builds up over several ticks and senders can fire more than once. with a decay of 0.8, an input that keeps arriving every tick settles at about 1 / (1 - 0.8) = 5 times its one-tick amount. so a neuron needs roughly 361 / 5 = 72 per tick to reach 361 this way, not the full 361 in one tick.
>
> caveat: leg motor neurons are further from the brain, so more of their inputs may come from neurons reached at the last hop, whose outgoing edges are not in the table. the low leg totals could partly be the 4-hop cutoff and not the real wiring.

prediction: with 5 hops, if the cutoff is the cause, the leg distribution should move towards the abdominal one (a higher 25th percentile, fewer with no incoming edges). if the legs stay low, the lead is in the wiring.

a different check:

In [32]:
ceiling = edges_jo.groupby('body_post')['weight'].sum()
print((ceiling >= 361).sum(), "of", len(ceiling), "=", (ceiling >= 361).mean())

24611 of 154594 = 0.1591976402706444


> at threshold 361, 15.9% of neurons have a total incoming weight of at least 361, so they can reach it in a single tick (about 24,611 of 154,594 neurons that have incoming edges).
>
> but 22.5% fired in the 361 run (34,819 / 154,662). more fired than the single-tick ceiling allows, because charge builds up over several ticks and senders can fire more than once. the decay (0.8) sets how much carries over from tick to tick, so the ceiling is not an upper bound on firing.